# 第 27 章 調査と分析の設計

Book 第 27 章に対応します。同じ圧縮強度の表を，材齢ごとに平均する場合と，同じ配合の二材齢を一組にしてから平均する場合で比べます。比較の対象や重みを変えると，どの問いへの答えになったかを考えます。

上から順に実行してください。前半は講義の実演です。後半の実習（約 20 分）で，自分が知りたいことを一つ選び，元の表がその問いに答えられるかを点検します。次の回にも使う分析用ノートとして，自分の計画と出力を保存します。

## 7 日の行と 28 日の行

コンクリートの配合を検討する担当者が，測る前に強度の見当をつけたいとします。ひとまず学習用の 631 行を読み，7 日と 28 日の強度を，その材齢の行一つずつを同じ重さで平均します。保留の強度はこのノートの計算・表示に使いません。

出力の `size` は材齢ごとの元の行数，`mean` は MPa で表した圧縮強度の平均です。図の点は原行，黒いひし形は平均です。83 行と 254 行の集まりには，同じ配合が同じ回数入っているでしょうか。

In [ ]:
from pathlib import Path

from IPython.display import display
import pandas as pd

paths = [
    Path("data/concrete-strength.csv"),
    Path("../data/concrete-strength.csv"),
    Path("concrete-strength.csv"),
]
local_path = next((path for path in paths if path.is_file()), None)
public_url = (
    "https://raw.githubusercontent.com/sohei-takagi/"
    "python-ds-course-materials/main/data/concrete-strength.csv"
)
concrete = pd.read_csv(local_path if local_path is not None else public_url)
train = concrete.loc[concrete["split"] == "train"]
two_ages = train.loc[train["age"].isin([7, 28])]
age_summary = two_ages.groupby("age")["strength"].agg(["size", "mean"])
age_summary.round(2)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(27)
fig, ax = plt.subplots(figsize=(6.8, 4.15), layout="constrained")
for position, (day, color) in enumerate([(7, "#1a657a"), (28, "#a15632")]):
    strengths = two_ages.loc[two_ages["age"] == day, "strength"]
    x = position + rng.uniform(-0.18, 0.18, size=len(strengths))
    ax.scatter(x, strengths, color=color, s=16, alpha=0.34, edgecolors="none")
    average = age_summary.loc[day, "mean"]
    ax.scatter(position, average, marker="D", color="#22272b",
               edgecolor="white", s=65, zorder=3, label="Mean" if position == 0 else None)
    ax.annotate(f"{average:.2f} MPa", (position, average),
                xytext=(position + 0.25, average + 1.2), textcoords="data",
                va="bottom", fontsize=10,
                arrowprops={"arrowstyle": "-", "color": "#687581", "lw": 0.8},
                bbox={"fc": "white", "ec": "none", "alpha": 0.88, "pad": 1.2})
ax.set(
    xlim=(-0.44, 1.64), ylim=(0, 90), xticks=[0, 1],
    xticklabels=[f"{day} days\n(n = {age_summary.loc[day, 'size']})" for day in [7, 28]],
    ylabel="Compressive strength (MPa)",
)
ax.set_axisbelow(True)
ax.grid(axis="y", color="#d8dfe1", linewidth=0.75)
ax.legend(loc="upper left")
plt.show()

## 原行と配合

`record_id` は原表の行番号です。`recipe_id` は七つの材料量が完全に一致する行をまとめた番号です。材料量は kg/m³，`age` は材齢（日），`strength` は測定された圧縮強度（MPa）です。

配合 115 では，材料量の異なる組がいくつ表示されますか。材齢 7 日から 28 日の記録の差を手で求めてみてください。この四行は同じ配合の記録ですが，配合番号から同じ試験体を追跡したかどうかは分かりません。

配合 130 では同じ材齢 28 日で強度が違う行が二つ残っています。二行を使って 28 日の平均を求めると何 MPa になりますか。符号や丸めを確かめるため，次の表と照合します。

In [ ]:
materials = [
    "cement", "slag", "fly_ash", "water", "superplasticizer",
    "coarse_aggregate", "fine_aggregate",
]
record_columns = ["record_id", "age", "strength"]
recipe_115 = train.loc[train["recipe_id"] == 115].sort_values("age")

print("Different sets of material amounts:", len(recipe_115[materials].drop_duplicates()))
recipe_115[record_columns].round({"strength": 2})

In [ ]:
recipe_130 = train.loc[
    (train["recipe_id"] == 130) & (train["age"] == 28), record_columns
]
recipe_130.round({"strength": 4})

## 両材齢を一組にする

まず配合 × 材齢で一行にします。表の `strength_mean` は同じ組の強度の平均，`n_records` は使った原行の件数です。次に `pivot()` で一配合を一行にし，`7` と `28` を二列にします。`NaN` は対応する組の記録がないことを示します。

両側の記録がそろった 73 配合で，28 日側から 7 日側を引きます。配合 115 と 130 の差を，もとの試験記録から求めた値と照合してください。

In [ ]:
recipe_age = two_ages.groupby(["recipe_id", "age"], as_index=False).agg(
    strength_mean=("strength", "mean"),
    n_records=("strength", "size"),
)
recipe_age.loc[recipe_age["recipe_id"].isin([115, 130])].round(4)

In [ ]:
by_recipe = recipe_age.pivot(
    index="recipe_id", columns="age", values="strength_mean",
)
by_recipe.head(8).round(2)

In [ ]:
paired = by_recipe.dropna(subset=[7, 28]).copy()
paired["difference"] = paired[28] - paired[7]
paired.loc[[115, 130], [7, 28, "difference"]].round(2)

73 配合だけに限って元の表を読み直すと，7 日が 79 行，28 日が 80 行あります。同じ配合・同じ材齢に複数行あるためです。7 日の 83 行と 28 日の 254 行を比べる計算，両材齢がある 73 配合の原行を比べる計算，配合の平均を一配合ずつ比べる計算では，どこで対象と重みが変わるかを確かめます。

散布図で 73 配合の差を読みます。横軸が 7 日の平均，縦軸は同じ配合の 28 日の平均です。点が対角線より上にあるとき，どちらの材齢が高いのでしょうか。平均差だけからは分からなかったことを一つ挙げます。

In [ ]:
matched_rows = two_ages.loc[two_ages["recipe_id"].isin(paired.index)]
row_means_in_pairs = matched_rows.groupby("age")["strength"].agg(["size", "mean"])
row_means_in_pairs.round(2)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(6.5, 5.7), layout="constrained")
limit = 10 * np.ceil(paired[[7, 28]].to_numpy().max() / 10)
ax.plot([0, limit], [0, limit], linestyle="--", linewidth=1.1,
        color="#687077", label="Equal strength")
ax.scatter(paired[7], paired[28], color="#4d6873", s=29, alpha=0.61,
           linewidths=0)
for recipe_id, color, offset in [
    (115, "#0e6584", (10, 5)), (130, "#a34929", (-93, 11)),
]:
    point = (paired.loc[recipe_id, 7], paired.loc[recipe_id, 28])
    ax.scatter(*point, color=color, s=70, edgecolor="white", linewidth=1.1, zorder=3)
    ax.annotate(f"Recipe {recipe_id}", point, xytext=offset,
                textcoords="offset points", fontsize=10, va="center", zorder=4,
                arrowprops={"arrowstyle": "-", "color": color},
                bbox={"fc": "white", "ec": "none", "alpha": 0.94, "pad": 1.5})
ax.set(xlim=(0, limit), ylim=(0, limit), aspect="equal",
       xlabel="7-day mean (MPa)", ylabel="28-day mean (MPa)")
ax.legend(loc="upper left")
plt.show()

In [ ]:
print("Paired recipes:", len(paired))
print("Recipes with a higher mean at 28 days:", (paired["difference"] > 0).sum())
paired["difference"].agg(["min", "median", "mean", "max"]).round(2)

## 何を知りたいか

- 手元の記録の特徴を説明するなら，集計に入る行や配合を指定して数えます。
- ある製造工程全体の平均を一部から見積もるなら，対象工程の配合一覧，抽出する単位とその方法，測定条件が必要です。
- 初めての配合を測定前に予測するなら，七つの材料量と指定材齢を使い，学習にない配合の実測と照合して誤差を求めます。
- 材料や試験条件を変えた効果を知りたいなら，何をどう変えて，どの対象をどちらの条件へ割り付けて測るかを決めます。

講師は新しい配合の予測を選びます。学習は 631 行・256 配合，候補を比べる検証は 208 行・86 配合，手順を決めてから使う保留は 191 行・86 配合です。次回は同じ学習部分で作った基準・直線・森を，同じ検証行の平均絶対誤差（MAE）などで比べます。配合を一つずつ同じ重さにして評価した場合も調べます。この章の集計は，学習部分の記録の記述です。

## 実習（約 20 分）

### 1．問いと予想を書く（約 4 分）

次の入口から一つ選び，知りたいことを一文にします。数字を見る前に，どの列にどんな記録がありそうか，予想の根拠も書きます。

| 入口 | セルで選ぶもの |
|---|---|
| 強度の記録を比較したい | `question_kind = "description"`。材齢 3・7・28・56 日から異なる二つを `chosen_ages` にリストで入れる。例は 3 日と 28 日，7 日と 56 日，28 日と 56 日。 |
| モデルで予測したい | `question_kind = "prediction"`。あとで誤差を詳しく読みたい材齢を，`chosen_band` に `"1to7"`（1〜7 日），`"8to28"`（8〜28 日），`"29plus"`（29 日以上）のいずれかとして入れる。 |
| 他の既習題材を使いたい | `question_kind = "other"`。下の資料を開き，元の表を点検するセルを一つ用意する。 |

**最初の問い・予想とその根拠**　（ここに記入）

### 2．表を絞る（約 6 分）

Concrete の入口を選んだ人は，次のセルで選択を設定します。`selected_rows = None` の右辺を書き換え，`train.loc[行の条件, read_columns]` を作ってください。記述の入口なら `train["age"].isin(…)` で二つの値を選べます。予測の入口では，表示用の下限・上限が `lower_age` と `upper_age` に入ります。既習の比較演算や `.between()` で，両端を含む行を選びます。29 日以上の上限は無限大です。

出力で，選択した範囲と元の行数，実際に入った値と欠けを確かめます。記述なら両方の材齢がある配合数も確認します。予測なら，何を予測前に知り，何を後で採点に使うかを列から説明します。ここで指定するのは，次回に詳しく見る評価の範囲です。学ぶときは引き続き学習側の 631 行を使います。

他の既習題材の場合は，先にデータの出典と一行の単位を読み，必要な列・個数・欠けが見えるセルを一つ，用意したノートから移します。自分のデータを使う場合も，同じ点を確認してください。

In [ ]:
question_kind = None  # "description", "prediction", or "other"
chosen_ages = None    # e.g., [3, 28] using two distinct ages from 3, 7, 28, 56
chosen_band = None    # "1to7", "8to28", or "29plus"

age_bands = {"1to7": (1, 7), "8to28": (8, 28), "29plus": (29, float("inf"))}
expected_rows = None

if question_kind == "description":
    allowed_ages = [3, 7, 28, 56]
    if (isinstance(chosen_ages, (list, tuple)) and len(chosen_ages) == 2
            and all(day in allowed_ages for day in chosen_ages)
            and chosen_ages[0] != chosen_ages[1]):
        expected_rows = train["age"].isin(chosen_ages)
        read_columns = ["record_id", "recipe_id", "age", "strength"]
        print("選択した材齢（日）", sorted(chosen_ages))
    else:
        print("3・7・28・56 日から異なる二つを chosen_ages に入れてください。")
elif question_kind == "prediction":
    if chosen_band in age_bands:
        lower_age, upper_age = age_bands[chosen_band]
        expected_rows = train["age"].between(lower_age, upper_age)
        read_columns = ["record_id", "recipe_id"] + materials + ["age", "strength"]
        print("詳しく確認する評価範囲", chosen_band)
    else:
        print("chosen_band に 1to7・8to28・29plus のいずれかを入れてください。")
elif question_kind == "other":
    print("選んだ資料を開き，表の点検セルをこの下へ追加してください。")
else:
    print("question_kind に description・prediction・other のいずれかを入れてください。")

if expected_rows is not None:
    selected_rows = None  # train.loc[自分で書いた行の条件, read_columns]
    if selected_rows is None:
        print("selected_rows の右辺に，選んだ条件で行と列を取り出す式を書いてください。")
    elif not isinstance(selected_rows, pd.DataFrame):
        print("行と複数列を含む表を取り出してください。")
    elif not selected_rows.index.equals(train.index[expected_rows]):
        print("選んだ材齢条件で学習表の行を絞り，該当行を保ってください。")
    elif selected_rows.columns.tolist() != read_columns:
        print("照合・入力・結果に必要な read_columns を列として指定してください。")
    else:
        print(f"学習表の確認範囲: {len(selected_rows)} 行・{selected_rows['recipe_id'].nunique()} 配合")
        display(selected_rows.head(3))
        display(selected_rows.isna().sum().rename("Missing values"))
        if question_kind == "description":
            both = selected_rows.groupby("recipe_id")["age"].nunique().eq(2).sum()
            print(f"選んだ二つの材齢がそろう配合: {both}")

### 他の題材のデータと出典

| 入り口 | 一行の単位 | 元にする資料 |
|---|---|---|
| 2020 年の市区町村人口を比べる | 2020 年の一市区町村 | [第 6 章のノート](https://colab.research.google.com/github/sohei-takagi/python-ds-course-materials/blob/main/practice/06_joins.ipynb) の読込セル。 [2020 年の CSV](https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/census-municipalities-2020.csv) と [出典](https://github.com/sohei-takagi/python-ds-course-materials/blob/main/data/census-sources.md) |
| 仮想抽出で平均を見積もる | 一羽のペンギンの記録 | [第 9 章のノート](https://colab.research.google.com/github/sohei-takagi/python-ds-course-materials/blob/main/practice/09_bootstrap.ipynb) ・ [出典](https://github.com/sohei-takagi/python-ds-course-materials/blob/main/data/penguins-sources.md) |
| 形状から米粒の品種を予測する | 一粒の米の形状記録 | [第 23 章のノート](https://colab.research.google.com/github/sohei-takagi/python-ds-course-materials/blob/main/practice/23_logistic_regression.ipynb) ・ [出典](https://github.com/sohei-takagi/python-ds-course-materials/blob/main/data/rice-sources.md) |
| 飲み物を割り付けたときの差を読む | 一人・一時点の測定 | [第 10 章のノート](https://colab.research.google.com/github/sohei-takagi/python-ds-course-materials/blob/main/practice/10_randomization.ipynb) ・ [出典](https://github.com/sohei-takagi/python-ds-course-materials/blob/main/data/mosquito-sources.md) |

市区町村人口を比べる場合は，2020 年の表だけを使うので，二時点を結合するセルは使いません。米粒で予測する人は，表の点検に学習側を使ってください。配布表に保留行も入っていますが，候補を決める前の集計には使いません。ペンギンの仮想抽出を扱うときには，一羽ごとの実測表を使って，どんな抽出を設定するかを書きます。

### 3．計画を書く（約 7 分）

- **問いと対象**　何を知りたいか，どの範囲の記録について答えるか。
- **表と一行**　使うファイル，一行の単位，使う列とその単位。
- **主な比較**　何と何を同じ条件で比べ，どんな表や数値を出すか。予測なら，測定前の入力と後で使う正解。
- **あとで確かめること**　どの条件や記録で確認し，どんな結果なら今の説明を見直すか。
- **問いの見直し**　読み込んだ表の件数・元の行・欠けを確認して，問いを変更したか，保ったか。理由を一つ書く。

**計画**　（ここに記入）

### 4．次に作る出力を選ぶ（約 3 分）

記述なら，講義セルの二材齢の指定と対応させる列を替え，配合ごとの違いが分かる一つの表か図を用意できます。予測なら，既習の基準とモデルを，同じ入力・同じ検証の行で比べる表が出発点です。保留への評価は，手順を固定してから行います。他の題材でも，問いに答える最初の図表を一つ決めます。

**次回までに作る図表・出典と利用条件・保存ファイル名**　（ここに記入）

この回で残すのは，予想・表の点検結果・計画・次に作る出力です。カーネルを再起動し，上から実行して，選択した値と出力が対応することを確認します。Colab では自分のドライブにコピーを保存し，ローカルでは分かりやすいファイル名を付けます。次回もそのコピーを開き，同じノートを更新してください。

### コンクリートの出典

Yeh, I. (1998). *Concrete Compressive Strength*. UCI Machine Learning Repository. Copyright Prof. I-Cheng Yeh. 教材は列名を短縮し，照合・分割に用いる列を追加しています。 [UCI データ DOI](https://doi.org/10.24432/C5PK67) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) ・ [教材データの出典と加工](https://github.com/sohei-takagi/python-ds-course-materials/blob/main/data/concrete-sources.md)